# Final Part 1: reference K-Means/K-NN hybrid

Authoritative source: origin/main at the commit in integration/provenance.json. All original Part 1 implementation cells are copied below. Default mode checks saved final predictions; set RUN_FULL=True and restart/run all to reproduce the existing 99-configuration research. Outputs are isolated; no original artifacts are overwritten.


In [1]:
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'part1-data').is_dir() and (p / 'part2-data').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Run inside the repository or assembled midterm folder with both data directories.')
RUN_FULL = False
REPRO_ROOT = ROOT / 'deliverables/reproduced'
REPRO_ROOT.mkdir(parents=True, exist_ok=True)
print('Mode:', 'full fresh reproduction' if RUN_FULL else 'read-only artifact verification; no training')



Mode: read-only artifact verification; no training


In [2]:
PROJECT_DIR = ROOT
WORK = REPRO_ROOT / 'part1_workflow'
WORK.mkdir(parents=True,exist_ok=True)
(WORK / 'writeup').mkdir(exist_ok=True)


In [3]:
if RUN_FULL:
    import pandas as pd
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, log_loss
else:
    print("Part 1 original cell 1 not retrained in verification mode.")


Part 1 original cell 1 not retrained in verification mode.


In [4]:
if RUN_FULL:
    PROJECT_DIR = ROOT
    PART1_DIR = PROJECT_DIR / "part1-data"
    PART2_DIR = PROJECT_DIR / "part2-data"

    reason_train = pd.read_csv(PART1_DIR / "part1_train.csv")
    reason_val = pd.read_csv(PART1_DIR / "part1_val.csv")
    reason_test = pd.read_csv(PART1_DIR / "part1_test.csv")

    outcome_train = pd.read_csv(PART2_DIR / "outcome_train.csv")
    outcome_val = pd.read_csv(PART2_DIR / "outcome_val.csv")
    outcome_test = pd.read_csv(PART2_DIR / "outcome_test.csv")

    pathway_catalog = pd.read_csv(PART1_DIR / "pathway_catalog.csv")
else:
    print("Part 1 original cell 2 not retrained in verification mode.")


Part 1 original cell 2 not retrained in verification mode.


## Part 1: Stopping reasons

Each trial can have several reason labels. In the training set, `-100` marks missing labels.

### Prepare the data

Use the 150 embedding columns and standardize them using the training data.

In [5]:
if RUN_FULL:
    labels = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    features = [f"emb_{i}" for i in range(150)]
    labeled = (reason_train[labels] != -100).all(axis=1)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(reason_train[features])
    X_val = scaler.transform(reason_val[features])
    X_test = scaler.transform(reason_test[features])

    y_train = reason_train.loc[labeled, labels]
    y_val = reason_val[labels]

    print("Labeled trials:", labeled.sum())
    print("Unlabeled trials:", (~labeled).sum())
else:
    print("Part 1 original cell 4 not retrained in verification mode.")


Part 1 original cell 4 not retrained in verification mode.


### Logistic regression baseline

Train on the labeled trials and evaluate on the validation set.

In [6]:
if RUN_FULL:
    reason_model = OneVsRestClassifier(
        LogisticRegression(max_iter=500, solver="liblinear", random_state=42)
    )
    reason_model.fit(X_train[labeled], y_train)
    val_pred = reason_model.predict(X_val)

    print("Sample-wise F1:", round(f1_score(y_val, val_pred, average="samples", zero_division=0), 4))
    print("Macro F1:", round(f1_score(y_val, val_pred, average="macro", zero_division=0), 4))

    per_label_f1 = f1_score(y_val, val_pred, average=None, zero_division=0)
    pd.Series(per_label_f1, index=labels, name="F1").round(4)
else:
    print("Part 1 original cell 6 not retrained in verification mode.")


Part 1 original cell 6 not retrained in verification mode.


### Part 1 experiments: K-means and GMM label recovery

Following HW3, cluster **all training embeddings**, including the unlabeled rows. Map each cluster to five independent reason frequencies using only the 157 visible training labels (never validation labels). K-means uses hard membership; GMM uses soft responsibilities. Smooth frequencies toward the labeled-training prior with two pseudo-observations so clusters without labeled members have a defined fallback. Threshold these frequencies to recover all 1,014 missing label vectors. Each stopped trial receives at least one reason (the highest-scoring reason if no threshold is met).

Fit five binary logistic regressions on the augmented training set, giving recovered rows less or equal weight than true labels. Keep known labels unchanged. Compare a fixed grid on validation sample-wise F1; use macro F1 only to break ties. Test embedding-only, added start year, and added text/drug-knowledge features. Clustering always uses only the reason embeddings, so feature ablations isolate the supervised stage. No test labels, IDs, or validation examples enter fitting. The validation set is small, so differences are exploratory rather than statistical evidence of improvement.

In [7]:
if RUN_FULL:
    import json
    import warnings
    import numpy as np
    from scipy import sparse
    from sklearn.cluster import KMeans
    from sklearn.mixture import GaussianMixture
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.exceptions import ConvergenceWarning
    from threadpoolctl import threadpool_limits

    RESULTS_DIR = WORK / "part1-results"
    RESULTS_DIR.mkdir(exist_ok=True)
    label_mask = labeled.to_numpy()
    y_known = y_train.to_numpy(dtype=int)
    y_validation = y_val.to_numpy(dtype=int)
    assert label_mask.sum() == 157 and (~label_mask).sum() == 1014
    assert reason_train.loc[~labeled, labels].eq(-100).all().all()
    assert np.isin(y_known, [0, 1]).all()
    assert not set(reason_train.ID) & set(reason_val.ID)
    assert not set(reason_train.ID) & set(reason_test.ID)
    assert not set(reason_val.ID) & set(reason_test.ID)

    label_summary = pd.DataFrame({
        "visible_positive": y_known.sum(axis=0),
        "visible_rate": y_known.mean(axis=0),
        "validation_rate": y_validation.mean(axis=0)
    }, index=labels)
    print("Training years (visible / masked):",
          reason_train.loc[labeled, "start_year"].mean(),
          reason_train.loc[~labeled, "start_year"].mean())
    display(label_summary.round(4))

    # All feature transformers learn only from training rows.
    year_scaler = StandardScaler()
    years = [year_scaler.fit_transform(reason_train[["start_year"]]),
             year_scaler.transform(reason_val[["start_year"]]),
             year_scaler.transform(reason_test[["start_year"]])]
    embedding_sets = [sparse.csr_matrix(x) for x in (X_train, X_val, X_test)]
    feature_sets = {
        "embeddings": tuple(embedding_sets),
        "embeddings_year": tuple(sparse.hstack([x, year], format="csr")
                                 for x, year in zip(embedding_sets, years))
    }
    reason_text_columns = ["criteria", "diseases_cleaned", "drugs", "icdcodes",
                           "drug_targets", "drug_actions", "drug_pathways"]
    def reason_text(frame):
        return frame[reason_text_columns].fillna("").agg(" ".join, axis=1)

    reason_vectorizer = TfidfVectorizer(min_df=3, max_features=3000,
                                       ngram_range=(1, 1), sublinear_tf=True)
    text_sets = [reason_vectorizer.fit_transform(reason_text(reason_train)),
                 reason_vectorizer.transform(reason_text(reason_val)),
                 reason_vectorizer.transform(reason_text(reason_test))]
    feature_sets["embeddings_year_text"] = tuple(
        sparse.hstack([x, year, text], format="csr")
        for x, year, text in zip(embedding_sets, years, text_sets)
    )
    print("Feature dimensions:", {name: data[0].shape[1] for name, data in feature_sets.items()})
else:
    print("Part 1 original cell 8 not retrained in verification mode.")


Part 1 original cell 8 not retrained in verification mode.


In [8]:
if RUN_FULL:
    def reason_metrics(truth, prediction):
        result = {
            "sample_f1": f1_score(truth, prediction, average="samples", zero_division=0),
            "macro_f1": f1_score(truth, prediction, average="macro", zero_division=0)
        }
        result.update(zip(labels, f1_score(truth, prediction, average=None, zero_division=0)))
        return result


    def threshold_reasons(probabilities, threshold=0.5):
        prediction = (probabilities >= threshold).astype(int)
        empty = prediction.sum(axis=1) == 0
        prediction[np.flatnonzero(empty), probabilities[empty].argmax(axis=1)] = 1
        return prediction


    def fit_reason_models(x, y, c, weights=None):
        # Explicit binary fits allow per-row weights for pseudo-labels.
        return [LogisticRegression(C=c, max_iter=1000, solver="liblinear", random_state=42)
                .fit(x, y[:, j], sample_weight=weights) for j in range(len(labels))]


    def reason_probabilities(models, x):
        return np.column_stack([model.predict_proba(x)[:, 1] for model in models])


    # Preserve the starter baseline exactly (including its original 0.5 decisions).
    records = [{"method": "starter", "features": "embeddings", "k": 0,
                "pseudo_threshold": 0.5, "pseudo_weight": 0.0, "C": 1.0,
                **reason_metrics(y_validation, val_pred)}]
    # Tune supervised controls using exactly the same final prediction rule.
    for feature_name, (train_x, val_x, _) in feature_sets.items():
        for c in (0.1, 1.0, 10.0):
            models = fit_reason_models(train_x[label_mask], y_known, c)
            pred = threshold_reasons(reason_probabilities(models, val_x))
            records.append({"method": "supervised", "features": feature_name, "k": 0,
                            "pseudo_threshold": 0.5, "pseudo_weight": 0.0, "C": c,
                            **reason_metrics(y_validation, pred)})

    # Cache recovered labels: changing the downstream feature set does not recluster.
    pseudo_labels = {}
    cluster_diagnostics = []
    prior = y_known.mean(axis=0)
    for method in ("kmeans", "gmm"):
        for k in (5, 10, 20):
            if method == "kmeans":
                clusterer = KMeans(n_clusters=k, init="k-means++", n_init=10,
                                   max_iter=300, tol=1e-4, random_state=42)
            else:
                clusterer = GaussianMixture(n_components=k, covariance_type="diag",
                                            n_init=3, init_params="kmeans", max_iter=200,
                                            tol=1e-3, reg_covar=1e-5, random_state=42)
            with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always", ConvergenceWarning)
                clusterer.fit(X_train)
            if method == "kmeans":
                membership = np.eye(k)[clusterer.labels_]
            else:
                membership = clusterer.predict_proba(X_train)
            known_membership = membership[label_mask]
            mass = known_membership.sum(axis=0)
            cluster_rates = (known_membership.T @ y_known + 2.0 * prior) / (mass[:, None] + 2.0)
            recovered_probabilities = membership[~label_mask] @ cluster_rates
            cluster_diagnostics.append({
                "method": method, "k": k, "iterations": int(clusterer.n_iter_),
                "converged": bool(getattr(clusterer, "converged_", clusterer.n_iter_ < 300)),
                "clusters_with_labeled_mass_below_1": int((mass < 1).sum()),
                "convergence_warnings": sum(issubclass(w.category, ConvergenceWarning) for w in caught)
            })
            for threshold in (0.3, 0.5):
                augmented = np.empty((len(reason_train), len(labels)), dtype=int)
                augmented[label_mask] = y_known
                augmented[~label_mask] = threshold_reasons(recovered_probabilities, threshold)
                assert np.array_equal(augmented[label_mask], y_known)
                pseudo_labels[(method, k, threshold)] = augmented
                for weight in (0.25, 1.0):
                    weights = np.where(label_mask, 1.0, weight)
                    for c in (0.1, 1.0):
                        models = fit_reason_models(feature_sets["embeddings"][0], augmented, c, weights)
                        pred = threshold_reasons(reason_probabilities(models, feature_sets["embeddings"][1]))
                        records.append({"method": method, "features": "embeddings", "k": k,
                                        "pseudo_threshold": threshold, "pseudo_weight": weight, "C": c,
                                        **reason_metrics(y_validation, pred)})


    def ranked_results(rows):
        # Stable order makes exact ties reproducible; sample-wise F1 is primary.
        return pd.DataFrame(rows).sort_values(["sample_f1", "macro_f1"], ascending=False, kind="stable")

    # Feature ablations reuse each method's best embedding-only recovery settings.
    for method in ("kmeans", "gmm"):
        best = ranked_results([r for r in records if r["method"] == method]).iloc[0]
        augmented = pseudo_labels[(method, int(best.k), float(best.pseudo_threshold))]
        weights = np.where(label_mask, 1.0, best.pseudo_weight)
        for feature_name in ("embeddings_year", "embeddings_year_text"):
            train_x, val_x, _ = feature_sets[feature_name]
            for c in (0.1, 1.0, 10.0):
                models = fit_reason_models(train_x, augmented, c, weights)
                pred = threshold_reasons(reason_probabilities(models, val_x))
                records.append({"method": method, "features": feature_name, "k": int(best.k),
                                "pseudo_threshold": best.pseudo_threshold,
                                "pseudo_weight": best.pseudo_weight, "C": c,
                                **reason_metrics(y_validation, pred)})

    experiments = ranked_results(records).reset_index(drop=True)
    experiments.to_csv(RESULTS_DIR / "experiments.csv", index=False)
    diagnostics = pd.DataFrame(cluster_diagnostics)
    diagnostics.to_csv(RESULTS_DIR / "cluster_diagnostics.csv", index=False)
    comparison = experiments.groupby(["method", "features"], sort=False).head(1)
    display(comparison.round(4))
    display(diagnostics)
    print("Validation configurations evaluated:", len(experiments))
else:
    print("Part 1 original cell 9 not retrained in verification mode.")


Part 1 original cell 9 not retrained in verification mode.


### Select and audit the semi-supervised model

Choose the best **K-means/GMM augmented classifier** by validation sample-wise F1, using macro F1 to break ties. Retain the strongest supervised control for an honest comparison even if augmentation is worse. Do not refit on validation labels: those labels remain available for later Part 2 error analysis. Export recovered training labels, validation predictions, and test predictions with explicit provenance so true labels and estimates can be distinguished when joining on trial ID.

The test CSV is used only for applying fitted transformations and generating predictions. No Kaggle score is used for model selection.

In [9]:
if RUN_FULL:
    selected = experiments[experiments.method.isin(["kmeans", "gmm"])].iloc[0]
    selected_y = pseudo_labels[(selected.method, int(selected.k), float(selected.pseudo_threshold))]
    selected_train_x, selected_val_x, selected_test_x = feature_sets[selected.features]
    selected_models = fit_reason_models(
        selected_train_x, selected_y, float(selected.C),
        np.where(label_mask, 1.0, float(selected.pseudo_weight))
    )
    selected_val_pred = threshold_reasons(reason_probabilities(selected_models, selected_val_x))
    selected_test_pred = threshold_reasons(reason_probabilities(selected_models, selected_test_x))
    assert np.isclose(reason_metrics(y_validation, selected_val_pred)["sample_f1"], selected.sample_f1)
    assert np.isin(selected_y, [0, 1]).all()
    assert np.array_equal(selected_y[label_mask], y_known)

    recovered_training = reason_train[["ID"]].copy()
    recovered_training[labels] = selected_y
    recovered_training["label_source"] = np.where(label_mask, "observed_train", "pseudo_train")
    recovered_training.to_csv(RESULTS_DIR / "recovered_training_labels.csv", index=False)
    validation_predictions = reason_val[["ID"]].copy()
    validation_predictions[labels] = selected_val_pred
    validation_predictions["label_source"] = "predicted_validation"
    validation_predictions.to_csv(RESULTS_DIR / "validation_predictions.csv", index=False)
    test_predictions = reason_test[["ID"]].copy()
    test_predictions[labels] = selected_test_pred
    test_predictions["label_source"] = "predicted_test"
    test_predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
    # A ready-to-join reason table for Part 2: use true validation reasons when available.
    observed_validation = reason_val[["ID", *labels]].copy()
    observed_validation["label_source"] = "observed_validation"
    reason_lookup = pd.concat([recovered_training, observed_validation, test_predictions], ignore_index=True)
    assert reason_lookup.ID.is_unique
    reason_lookup.to_csv(RESULTS_DIR / "reason_lookup.csv", index=False)

    label_summary["recovered_positive"] = selected_y[~label_mask].sum(axis=0)
    label_summary["recovered_rate"] = selected_y[~label_mask].mean(axis=0)
    label_summary.to_csv(RESULTS_DIR / "label_summary.csv", index_label="reason")
    selection_info = {name: selected[name] for name in
                      ("method", "features", "k", "pseudo_threshold", "pseudo_weight", "C",
                       "sample_f1", "macro_f1")}
    selection_info["k"] = int(selection_info["k"])
    selection_info["random_state"] = 42
    selection_info["validation_configurations"] = len(experiments)
    (RESULTS_DIR / "selected_model.json").write_text(json.dumps(selection_info, indent=2) + "\n")
    print("Selected semi-supervised configuration:", selection_info)
    display(label_summary.round(4))

    # Generate the report table directly from measured results.
    baseline = experiments[experiments.method == "starter"].iloc[0]
    best_supervised = experiments[experiments.method == "supervised"].iloc[0]
    best_kmeans = experiments[experiments.method == "kmeans"].iloc[0]
    best_gmm = experiments[experiments.method == "gmm"].iloc[0]
    table_rows = [r"\begin{tabular}{lrrrr}", r"\toprule",
                  'Metric & Starter & Sup. & K-means & GMM \\\\', r"\midrule"]
    for metric, title in [("sample_f1", "Sample F1"), ("macro_f1", "Macro F1"), *[(x, x) for x in labels]]:
        values = [row[metric] for row in (baseline, best_supervised, best_kmeans, best_gmm)]
        table_rows.append(title + " & " + " & ".join(f"{value:.4f}" for value in values) + ' \\\\')
    table_rows.extend([r"\bottomrule", r"\end{tabular}"])
    (WORK / "writeup" / "part1_results.tex").write_text("\n".join(table_rows) + "\n")

    feature_rows = [r"\begin{tabular}{lrrr}", r"\toprule",
                    "Features & Supervised & K-means & GMM " + chr(92) * 2, r"\midrule"]
    for name, title in [("embeddings", "E"), ("embeddings_year", "E+Y"),
                        ("embeddings_year_text", "E+Y+T")]:
        values = [experiments[(experiments.method == method) & (experiments.features == name)].iloc[0].sample_f1
                  for method in ("supervised", "kmeans", "gmm")]
        feature_rows.append(title + " & " + " & ".join(f"{value:.4f}" for value in values) + " " + chr(92) * 2)
    feature_rows.extend([r"\bottomrule", r"\end{tabular}"])
    (WORK / "writeup" / "part1_feature_results.tex").write_text("\n".join(feature_rows) + "\n")

    import platform
    from importlib.metadata import version
    environment = {name: version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "nbclient"]}
    environment["python"] = platform.python_version()
    (RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
else:
    print("Part 1 original cell 11 not retrained in verification mode.")


Part 1 original cell 11 not retrained in verification mode.


### Inspect the selected K-means clusters with PCA and observed reasons

These diagnostics examine the selected K-means label-recovery model; they do **not** change its predictions or tune new settings. Refit its deterministic training-only clustering configuration, then fit PCA on the same standardized training embeddings. PCA is only a visualization here: clustering still uses all 150 standardized dimensions. The first two components can discard substantial information, so apparent overlap in 2D does not establish overlap in the original space.

1. Color the PCA projection by cluster and mark observed-label trials with circles; crosses show projected centroids.
2. Show each observed reason separately so multi-label trials can appear in several panels. Gray points have hidden labels, not known-negative labels.
3. Compare raw observed positive frequencies with the smoothed probabilities actually used for recovery. Report counts alongside rates: tiny labeled groups are unreliable.
4. Inspect Funding explicitly and export representative trials nearest each centroid in the **original 150-dimensional space**. Original stopping explanations are unavailable; diseases and drugs provide context, not direct evidence of the stopping explanation.

In [10]:
if RUN_FULL:
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    from matplotlib.colors import BoundaryNorm
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score

    FIGURES_DIR = WORK / "writeup" / "figures"
    FIGURES_DIR.mkdir(exist_ok=True)
    inspection_k = int(best_kmeans.k)
    inspection_model = KMeans(n_clusters=inspection_k, init="k-means++", n_init=10,
                              max_iter=300, tol=1e-4, random_state=42)
    with threadpool_limits(limits=1):
        inspection_clusters = inspection_model.fit_predict(X_train)
        pca_inspection = PCA(svd_solver="full")
        projected = pca_inspection.fit_transform(X_train)
    projected_centroids = pca_inspection.transform(inspection_model.cluster_centers_)
    explained = pca_inspection.explained_variance_ratio_
    cumulative = explained.cumsum()
    projection_summary = {
        "cluster_method": "kmeans", "k": inspection_k,
        "pc1_variance_fraction": float(explained[0]),
        "pc2_variance_fraction": float(explained[1]),
        "first_two_variance_fraction": float(cumulative[1]),
        "components_for_90_percent": int(np.searchsorted(cumulative, 0.90) + 1),
        "components_for_95_percent": int(np.searchsorted(cumulative, 0.95) + 1),
        "silhouette_original_space_sample_1000": float(silhouette_score(
            X_train, inspection_clusters, sample_size=1000, random_state=42))
    }
    (RESULTS_DIR / "pca_inspection_summary.json").write_text(json.dumps(projection_summary, indent=2) + "\n")
    pd.DataFrame({"component": np.arange(1, len(explained) + 1),
                  "variance_fraction": explained, "cumulative_fraction": cumulative}).to_csv(
                      RESULTS_DIR / "pca_explained_variance.csv", index=False)
    coordinates = reason_train[["ID"]].copy()
    coordinates["cluster"] = inspection_clusters
    coordinates["observed_labels"] = label_mask
    coordinates["PC1"] = projected[:, 0]
    coordinates["PC2"] = projected[:, 1]
    coordinates.to_csv(RESULTS_DIR / "pca_coordinates.csv", index=False)
    print("PCA projection summary:", projection_summary)


    def save_inspection_figure(fig, name):
        fig.savefig(FIGURES_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
        fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")
        plt.show()
        plt.close(fig)


    axis_labels = [f"PC1 ({explained[0]:.1%} variance)", f"PC2 ({explained[1]:.1%} variance)"]
    fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
    cluster_cmap = plt.get_cmap("tab10", inspection_k)
    norm = BoundaryNorm(np.arange(-0.5, inspection_k + 0.5), inspection_k)
    points = ax.scatter(projected[~label_mask, 0], projected[~label_mask, 1],
                        c=inspection_clusters[~label_mask], cmap=cluster_cmap, norm=norm,
                        s=17, alpha=0.45, edgecolors="none", label="Hidden labels")
    ax.scatter(projected[label_mask, 0], projected[label_mask, 1],
               c=inspection_clusters[label_mask], cmap=cluster_cmap, norm=norm,
               s=46, edgecolors="black", linewidths=0.8, label="Observed labels")
    ax.scatter(projected_centroids[:, 0], projected_centroids[:, 1], marker="x",
               color="black", s=85, linewidths=2, label="Projected centroids")
    for cluster, (cx, cy) in enumerate(projected_centroids[:, :2]):
        ax.annotate(str(cluster), (cx, cy), xytext=(5, 6), textcoords="offset points")
    fig.colorbar(points, ax=ax, ticks=range(inspection_k), label="K-means cluster")
    ax.set(xlabel=axis_labels[0], ylabel=axis_labels[1],
           title=f"Training clusters projected with PCA ({cumulative[1]:.1%} of variance shown)")
    ax.legend(handles=[
        Line2D([], [], marker="o", linestyle="none", markerfacecolor="0.6", markeredgecolor="none",
               markersize=5, alpha=0.45, label="Hidden labels"),
        Line2D([], [], marker="o", linestyle="none", markerfacecolor="0.6", markeredgecolor="black",
               markersize=7, label="Observed labels"),
        Line2D([], [], marker="x", linestyle="none", color="black", markersize=8,
               label="Projected centroids")], loc="best")
    save_inspection_figure(fig, "part1_pca_clusters")

    fig, axes = plt.subplots(2, 3, figsize=(12, 7.5), sharex=True, sharey=True,
                             constrained_layout=True)
    for j, (label, ax) in enumerate(zip(labels, axes.flat)):
        positive = label_mask & reason_train[label].eq(1).to_numpy()
        negative = label_mask & reason_train[label].eq(0).to_numpy()
        ax.scatter(projected[~label_mask, 0], projected[~label_mask, 1], s=9,
                   color="#c9c9c9", alpha=0.45, edgecolors="none", label="Hidden labels")
        ax.scatter(projected[negative, 0], projected[negative, 1], s=18,
                   color="#434343", alpha=0.65, label="Observed negative")
        ax.scatter(projected[positive, 0], projected[positive, 1], s=45, marker="^",
                   color="#b43663", edgecolors="black", linewidths=0.4, label="Observed positive")
        ax.set(title=f"{label}: {positive.sum()} observed positives", xlabel=axis_labels[0], ylabel=axis_labels[1])
    handles, legend_labels = axes.flat[0].get_legend_handles_labels()
    axes.flat[-1].axis("off")
    axes.flat[-1].legend(handles, legend_labels, loc="center")
    fig.suptitle("Observed reasons in the same PCA projection (labels can overlap)")
    save_inspection_figure(fig, "part1_pca_reasons")

    cluster_sizes = np.bincount(inspection_clusters, minlength=inspection_k)
    known_cluster_sizes = np.bincount(inspection_clusters[label_mask], minlength=inspection_k)
    reason_counts = np.zeros((inspection_k, len(labels)), dtype=int)
    for cluster in range(inspection_k):
        reason_counts[cluster] = y_known[inspection_clusters[label_mask] == cluster].sum(axis=0)
    raw_rates = np.divide(reason_counts, known_cluster_sizes[:, None],
                          out=np.full(reason_counts.shape, np.nan), where=known_cluster_sizes[:, None] > 0)
    smoothed_rates = (reason_counts + 2.0 * prior) / (known_cluster_sizes[:, None] + 2.0)
    cluster_label_decisions = threshold_reasons(smoothed_rates, float(best_kmeans.pseudo_threshold))
    recovered_for_inspection = pseudo_labels[("kmeans", inspection_k, float(best_kmeans.pseudo_threshold))]
    assert np.array_equal(recovered_for_inspection[~label_mask],
                          cluster_label_decisions[inspection_clusters[~label_mask]])
    cluster_summary = pd.DataFrame({"cluster": range(inspection_k), "total": cluster_sizes,
                                    "observed": known_cluster_sizes,
                                    "hidden": cluster_sizes - known_cluster_sizes})
    for name, values in [("cluster_reason_counts", reason_counts), ("cluster_reason_rates", raw_rates),
                         ("cluster_reason_smoothed", smoothed_rates)]:
        pd.DataFrame(values, columns=labels).rename_axis("cluster").to_csv(RESULTS_DIR / f"{name}.csv")
    cluster_summary.to_csv(RESULTS_DIR / "cluster_sizes.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    for panel, (ax, matrix, title) in enumerate(zip(axes, [raw_rates, smoothed_rates],
            ["Observed frequency (positives / observed trials)",
             f"Smoothed recovery scores (* meets {float(best_kmeans.pseudo_threshold):g} threshold)"])):
        heatmap = ax.imshow(np.ma.masked_invalid(matrix), vmin=0, vmax=1, cmap="Blues", aspect="auto")
        ax.set_xticks(range(len(labels)), labels, rotation=35, ha="right")
        ax.set_yticks(range(inspection_k),
                      [f"C{k}: n={cluster_sizes[k]}, L={known_cluster_sizes[k]}" for k in range(inspection_k)])
        ax.set(title=title, ylabel="Cluster: total n, observed L")
        for cluster in range(inspection_k):
            for j in range(len(labels)):
                value = matrix[cluster, j]
                if panel == 0:
                    annotation = f"{reason_counts[cluster, j]}/{known_cluster_sizes[cluster]}" if np.isfinite(value) else "N/A"
                else:
                    annotation = f"{value:.2f}" + ("*" if value >= float(best_kmeans.pseudo_threshold) else "")
                ax.text(j, cluster, annotation, ha="center", va="center",
                        color="white" if value > 0.55 else "black", fontsize=9)
    fig.colorbar(heatmap, ax=axes, label="Probability / observed fraction", shrink=0.8)
    save_inspection_figure(fig, "part1_cluster_reason_heatmap")

    funding_j = labels.index("Funding")
    funding_inspection = cluster_summary.copy()
    funding_inspection["observed_funding_positive"] = reason_counts[:, funding_j]
    funding_inspection["raw_funding_rate"] = raw_rates[:, funding_j]
    funding_inspection["smoothed_funding_score"] = smoothed_rates[:, funding_j]
    funding_inspection["funding_pseudo_positive"] = [
        int(recovered_for_inspection[(inspection_clusters == k) & ~label_mask, funding_j].sum())
        for k in range(inspection_k)]
    funding_inspection.to_csv(RESULTS_DIR / "funding_cluster_inspection.csv", index=False)
    print("Funding cluster inspection (true labels only in observed counts):")
    display(funding_inspection.round(4))

    # Nearest examples are selected in the fitted 150-dimensional clustering space.
    distance_to_centroid = np.linalg.norm(
        X_train - inspection_model.cluster_centers_[inspection_clusters], axis=1)
    representatives = []
    for cluster in range(inspection_k):
        members = np.flatnonzero(inspection_clusters == cluster)
        nearest = members[np.argsort(distance_to_centroid[members], kind="stable")[:3]]
        for index in nearest:
            row = reason_train.iloc[index]
            representatives.append({
                "cluster": cluster, "ID": row.ID, "start_year": row.start_year,
                "distance_to_centroid": distance_to_centroid[index],
                "label_source": "observed" if label_mask[index] else "hidden",
                "observed_reasons": " ".join(label for label in labels if row[label] == 1)
                                    if label_mask[index] else "unavailable",
                "diseases": str(row.diseases_cleaned)[:160] if pd.notna(row.diseases_cleaned) else "",
                "drugs": str(row.drugs)[:100] if pd.notna(row.drugs) else ""
            })
    representative_trials = pd.DataFrame(representatives)
    representative_trials.to_csv(RESULTS_DIR / "cluster_representatives.csv", index=False)
    display(representative_trials.round(3))

    # Record the plotting dependency alongside the original environment audit.
    environment["matplotlib"] = version("matplotlib")
    _ = (RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
else:
    print("Part 1 original cell 13 not retrained in verification mode.")


Part 1 original cell 13 not retrained in verification mode.


### What the inspection shows

- **PCA is a limited view:** PC1 explains 16.7% and PC2 7.4% of training variance (24.2% together). It takes 64 components to retain 90% and 92 to retain 95%. Apparent overlap in these plots is not evidence that the original 150-dimensional clusters are inseparable. The original-space silhouette on a reproducible 1,000-trial sample is 0.1704; this is a descriptive geometry measure, not a label-recovery accuracy score.
- **Some clusters have clear observed-label associations:** cluster 0 has 14/25 Safety-positive observed trials; cluster 5 has 14/18 Efficacy-positive trials. Several other clusters are dominated by Recruitment. Cluster 0 also has 8/25 Other positives, illustrating why cluster-to-reason mapping must allow several labels.
- **Funding suppression is directly explained by the recovery rule:** all eight observed Funding positives lie in clusters 0, 1, 3, and 7 (counts 2, 1, 1, and 4). The largest raw frequency is 4/18 = 0.2222 in cluster 7; smoothing reduces it to 0.2051, below the selected 0.3 threshold. Every cluster falls below that threshold and the highest-score fallback never chooses Funding. Therefore all 1,014 recovered Funding labels are zero. This is a limitation of cluster-level probabilities plus thresholding, not proof that the embeddings lack Funding information.
- **Small labeled groups limit confidence:** clusters 4 and 6 have only 3 and 4 observed trials. Their frequencies should not be treated as precise probabilities for the larger unlabeled groups.

Possible follow-ups are label-specific recovery thresholds, local K-NN estimates, confidence filtering, or soft-label training. These have **not** been tested here. No classifier, pseudo-label, model selection, or submission was changed by this inspection.

### Controlled follow-up: lower only the Funding recovery threshold

**Predefined grid:** Funding thresholds 0.30 (control), 0.20, 0.15, and 0.10. Keep the selected K-means model, all other reason pseudo-labels, embedding-only features, pseudo-label weight 0.25, logistic-regression C=0.1, and final classifier threshold 0.5 fixed. These thresholds apply to **recovering training labels**, not thresholding validation predictions. Known training labels are never changed.

Compare validation Funding precision/recall/F1, confusion counts, sample-wise F1, macro F1, and the number of recovered Funding positives. Only four validation trials are Funding-positive, so results are exploratory. Compare the control against the original selected method as a reproducibility check. Save follow-up artifacts separately; the original submission and 70-configuration results remain unchanged. K-NN has not been implemented in this experiment.

In [11]:
if RUN_FULL:
    from sklearn.metrics import precision_score, recall_score

    FUNDING_DIR = RESULTS_DIR / "funding-threshold-experiment"
    FUNDING_DIR.mkdir(exist_ok=True)
    funding_thresholds = (0.30, 0.20, 0.15, 0.10)
    funding_index = labels.index("Funding")
    other_label_indices = [j for j in range(len(labels)) if j != funding_index]
    control_y = pseudo_labels[("kmeans", inspection_k, float(best_kmeans.pseudo_threshold))]
    assert best_kmeans.features == "embeddings"
    assert float(best_kmeans.C) == 0.1 and float(best_kmeans.pseudo_weight) == 0.25
    trial_funding_scores = smoothed_rates[inspection_clusters, funding_index]
    funding_records = []
    funding_candidate_predictions = {}
    for threshold in funding_thresholds:
        candidate_y = control_y.copy()
        candidate_y[~label_mask, funding_index] = (trial_funding_scores[~label_mask] >= threshold).astype(int)
        assert np.array_equal(candidate_y[label_mask], y_known)
        assert np.array_equal(candidate_y[:, other_label_indices], control_y[:, other_label_indices])
        models = fit_reason_models(feature_sets["embeddings"][0], candidate_y, 0.1,
                                   np.where(label_mask, 1.0, 0.25))
        prediction = threshold_reasons(reason_probabilities(models, feature_sets["embeddings"][1]))
        funding_candidate_predictions[threshold] = prediction
        actual = y_validation[:, funding_index]
        predicted = prediction[:, funding_index]
        funding_records.append({
            "funding_threshold": threshold,
            "funding_pseudo_positives": int(candidate_y[~label_mask, funding_index].sum()),
            "funding_precision": precision_score(actual, predicted, zero_division=0),
            "funding_recall": recall_score(actual, predicted, zero_division=0),
            "funding_tp": int(((actual == 1) & (predicted == 1)).sum()),
            "funding_fp": int(((actual == 0) & (predicted == 1)).sum()),
            "funding_fn": int(((actual == 1) & (predicted == 0)).sum()),
            **reason_metrics(y_validation, prediction)
        })
        output = reason_val[["ID"]].copy()
        output[labels] = prediction
        output.to_csv(FUNDING_DIR / f"validation_threshold_{threshold:.2f}.csv", index=False)
        recovered_output = reason_train[["ID"]].copy()
        recovered_output["Funding"] = candidate_y[:, funding_index]
        recovered_output["label_source"] = np.where(label_mask, "observed_train", "pseudo_train")
        recovered_output.to_csv(FUNDING_DIR / f"training_funding_threshold_{threshold:.2f}.csv", index=False)

    funding_results = pd.DataFrame(funding_records)
    assert np.isclose(funding_results.iloc[0].sample_f1, best_kmeans.sample_f1)
    assert np.isclose(funding_results.iloc[0]["Funding"], best_kmeans["Funding"])
    funding_results.to_csv(FUNDING_DIR / "results.csv", index=False)
    display(funding_results[["funding_threshold", "funding_pseudo_positives", "funding_precision",
                             "funding_recall", "Funding", "funding_tp", "funding_fp", "funding_fn",
                             "sample_f1", "macro_f1"]].round(4))
    funding_best = funding_results.sort_values(["sample_f1", "macro_f1"], ascending=False,
                                              kind="stable").iloc[0]
    print("Best follow-up threshold by sample-wise F1, then macro F1:", funding_best.funding_threshold)
    print("Original submission and selected-model artifacts remain unchanged.")

else:
    print("Part 1 original cell 16 not retrained in verification mode.")


Part 1 original cell 16 not retrained in verification mode.


### Funding threshold experiment: interpretation

Lowering the recovery threshold from 0.30 to 0.20, 0.15, or 0.10 assigns Funding to the same 112 hidden-label trials in cluster 7. No other cluster's smoothed Funding score reaches 0.10, so these three candidates are identical. All observed labels and other pseudo-labels remain unchanged.

The control classifier identifies 1 of 4 Funding-positive validation trials and has no Funding false positives (precision 1.0, recall 0.25, F1 0.40). Each lower-threshold classifier still identifies only 1 of the 4, but adds 3 false positives (precision 0.25, recall 0.25, F1 0.25). Sample-wise F1 falls from 0.8547 to 0.8419; macro F1 falls from 0.6923 to 0.6623. We therefore retain threshold 0.30 and the original model/submission. This result applies to this fixed clustering, representation, and weighting setting; only four Funding positives limit the strength of the conclusion.

This illustrates a risk of cluster-wide recovery: a lower threshold promotes an entire group, rather than identifying which members plausibly have Funding reasons. A local neighbor-based estimate is a motivated follow-up, not a proven improvement. K-NN has not been tested here.

### Local label recovery with K-NN, and a K-means/K-NN hybrid

**Predefined grid:** K = 3, 5, 9, or 15 labeled neighbors, using uniform voting or inverse-distance weighting. Neighbors are searched among the **157 observed training trials only**, in the original 150-dimensional standardized embedding space. Unlabeled trials never become reference neighbors, and validation trials are never used as neighbors or for fitting.

Compare three uses of the same neighbor settings:

- **Direct K-NN (supervised control):** estimate validation reason probabilities directly from the labeled neighbors. This alone does not meet the unsupervised-method requirement.
- **K-NN recovery:** estimate missing training labels from neighbors, then fit the same weighted logistic regressions as before. This is a comparison to cluster-based recovery, not a substitute for the required unsupervised experiment.
- **K-means/K-NN hybrid:** average each reason's K-NN score and its smoothed cluster score equally, recover missing labels, and fit weighted logistic regressions. This explicitly retains unsupervised clustering in the recovery process.

For both recovery methods, hold the pseudo-label threshold at 0.3, pseudo-label weight at 0.25, classifier C at 0.1, and embedding-only features fixed. Preserve observed labels; final predictions use 0.5 with the existing highest-score fallback. Include a labeled-only logistic-regression control with C=0.1 to isolate augmentation effects. The original C=1 starter and selected K-means results remain reference scores. No new feature, threshold, C, or blend-weight search is performed.

Select the best augmented candidate by validation sample-wise F1, then macro F1, with stable grid order for remaining ties. Export its test predictions separately without overwriting the existing selected-model artifacts or original submission. Neighbor vote fractions and hybrid scores are estimates, not calibrated probabilities. This is further exploratory reuse of the small validation set.

In [12]:
if RUN_FULL:
    from sklearn.neighbors import KNeighborsClassifier

    KNN_DIR = RESULTS_DIR / "knn-experiment"
    KNN_DIR.mkdir(exist_ok=True)
    knn_records = []
    knn_recovered_candidates = {}
    knn_augmented_models = {}
    knn_validation_predictions = {}


    def neighbor_reason_scores(model, query):
        probabilities = model.predict_proba(query)
        return np.column_stack([
            probability[:, np.flatnonzero(classes == 1)[0]]
            for probability, classes in zip(probabilities, model.classes_)
        ])


    def knn_record(method, k, weighting, prediction, augmented=None):
        actual = y_validation[:, labels.index("Funding")]
        predicted = prediction[:, labels.index("Funding")]
        record = {
            "method": method, "neighbors": k, "neighbor_weighting": weighting,
            "funding_precision": precision_score(actual, predicted, zero_division=0),
            "funding_recall": recall_score(actual, predicted, zero_division=0),
            "funding_tp": int(((actual == 1) & (predicted == 1)).sum()),
            "funding_fp": int(((actual == 0) & (predicted == 1)).sum()),
            "funding_fn": int(((actual == 1) & (predicted == 0)).sum()),
            **reason_metrics(y_validation, prediction)
        }
        for j, label in enumerate(labels):
            record[f"pseudo_{label}"] = int(augmented[~label_mask, j].sum()) if augmented is not None else 0
        return record


    matched_control_models = fit_reason_models(feature_sets["embeddings"][0][label_mask], y_known, 0.1)
    matched_control_prediction = threshold_reasons(
        reason_probabilities(matched_control_models, feature_sets["embeddings"][1]))
    knn_records.append(knn_record("supervised_lr_C0.1", 0, "none", matched_control_prediction))
    cluster_scores_for_hidden = smoothed_rates[inspection_clusters[~label_mask]]
    for k in (3, 5, 9, 15):
        for weighting in ("uniform", "distance"):
            neighbors = KNeighborsClassifier(n_neighbors=k, weights=weighting,
                                             metric="euclidean", algorithm="brute", n_jobs=1)
            neighbors.fit(X_train[label_mask], y_known)
            with threadpool_limits(limits=1):
                hidden_neighbor_scores = neighbor_reason_scores(neighbors, X_train[~label_mask])
                direct_val_prediction = threshold_reasons(neighbor_reason_scores(neighbors, X_val))
            knn_records.append(knn_record("direct_knn", k, weighting, direct_val_prediction))
            knn_validation_predictions[("direct_knn", k, weighting)] = direct_val_prediction
            for method in ("knn_recovery", "kmeans_knn_hybrid"):
                recovery_scores = hidden_neighbor_scores if method == "knn_recovery" else (
                    0.5 * hidden_neighbor_scores + 0.5 * cluster_scores_for_hidden)
                augmented = np.empty((len(reason_train), len(labels)), dtype=int)
                augmented[label_mask] = y_known
                augmented[~label_mask] = threshold_reasons(recovery_scores, 0.3)
                assert np.array_equal(augmented[label_mask], y_known)
                assert np.isin(augmented, [0, 1]).all()
                models = fit_reason_models(feature_sets["embeddings"][0], augmented, 0.1,
                                           np.where(label_mask, 1.0, 0.25))
                prediction = threshold_reasons(reason_probabilities(models, feature_sets["embeddings"][1]))
                key = (method, k, weighting)
                knn_records.append(knn_record(method, k, weighting, prediction, augmented))
                knn_recovered_candidates[key] = augmented
                knn_augmented_models[key] = models
                knn_validation_predictions[key] = prediction

    knn_results = pd.DataFrame(knn_records).sort_values(["sample_f1", "macro_f1"],
                                                      ascending=False, kind="stable").reset_index(drop=True)
    knn_results.to_csv(KNN_DIR / "results.csv", index=False)
    for (method, k, weighting), prediction in knn_validation_predictions.items():
        output = reason_val[["ID"]].copy()
        output[labels] = prediction
        output.to_csv(KNN_DIR / f"validation_{method}_k{k}_{weighting}.csv", index=False)
    print("Best configuration within each method:")
    display(knn_results.groupby("method", sort=False).head(1)[
        ["method", "neighbors", "neighbor_weighting", "sample_f1", "macro_f1", "Funding",
         "funding_precision", "funding_recall", "funding_tp", "funding_fp", "pseudo_Funding"]].round(4))
    print("All 25 predefined follow-up configurations:")
    display(knn_results[["method", "neighbors", "neighbor_weighting", "sample_f1", "macro_f1", "Funding"]].round(4))

    best_knn_augmented = knn_results[knn_results.method.isin(["knn_recovery", "kmeans_knn_hybrid"])].iloc[0]
    knn_key = (best_knn_augmented.method, int(best_knn_augmented.neighbors), best_knn_augmented.neighbor_weighting)
    best_knn_recovered = reason_train[["ID"]].copy()
    best_knn_recovered[labels] = knn_recovered_candidates[knn_key]
    best_knn_recovered["label_source"] = np.where(label_mask, "observed_train", "pseudo_train")
    best_knn_recovered.to_csv(KNN_DIR / "best_augmented_training_labels.csv", index=False)
    best_knn_test_pred = threshold_reasons(reason_probabilities(knn_augmented_models[knn_key], feature_sets["embeddings"][2]))
    knn_test_output = reason_test[["ID"]].copy()
    knn_test_output["Predicted"] = [" ".join(label for label, value in zip(labels, row) if value)
                                    for row in best_knn_test_pred]
    assert knn_test_output.ID.equals(reason_test.ID) and knn_test_output.Predicted.str.len().gt(0).all()
    knn_test_output.to_csv(KNN_DIR / "best_augmented_candidate_submission.csv", index=False)
    knn_selection = {"method": knn_key[0], "neighbors": knn_key[1], "neighbor_weighting": knn_key[2],
                     "sample_f1": float(best_knn_augmented.sample_f1),
                     "macro_f1": float(best_knn_augmented.macro_f1),
                     "configurations": len(knn_results), "pseudo_threshold": 0.3,
                     "pseudo_weight": 0.25, "C": 0.1, "features": "embeddings",
                     "cluster_score_weight": 0.5 if knn_key[0] == "kmeans_knn_hybrid" else 0.0,
                     "overwrites_original_selection": False}
    _ = (KNN_DIR / "best_augmented_candidate.json").write_text(json.dumps(knn_selection, indent=2) + "\n")
    print("Best augmented candidate:", knn_selection)
    print("Reference sample-wise F1: starter =", baseline.sample_f1, "; selected K-means =", best_kmeans.sample_f1)

    # Regenerate the follow-up report table from measured results.
    knn_table_methods = ["direct_knn", "knn_recovery", "kmeans_knn_hybrid"]
    knn_table_best = [knn_results[knn_results.method == method].iloc[0] for method in knn_table_methods]
    knn_table_rows = [r"\begin{tabular}{lrrr}", r"\toprule",
                      "Metric & Direct KNN & KNN+LR & Hybrid " + chr(92) * 2, r"\midrule"]
    for metric, title in [("sample_f1", "Sample F1"), ("macro_f1", "Macro F1"), *[(label, label) for label in labels]]:
        knn_table_rows.append(title + " & " + " & ".join(f"{row[metric]:.4f}" for row in knn_table_best) + " " + chr(92) * 2)
    knn_table_rows.extend([r"\bottomrule", r"\end{tabular}"])
    _ = (WORK / "writeup" / "part1_knn_results.tex").write_text("\n".join(knn_table_rows) + "\n")

else:
    print("Part 1 original cell 19 not retrained in verification mode.")


Part 1 original cell 19 not retrained in verification mode.


### K-NN follow-up: results and interpretation

The best local-recovery setting uses **3 uniformly weighted labeled neighbors**. Its augmented logistic regression scores sample-wise F1 0.8803 and macro F1 0.7873. Equal blending with the original K-means cluster scores improves sample-wise F1 to **0.8868**, with macro F1 **0.7792**. Both exceed the original K-means result (0.8547 / 0.6923); the hybrid also exceeds the starter (0.8654 / 0.7067) and the earlier strongest supervised control (0.8782 / 0.6990) on these validation data. The matched labeled-only logistic-regression control at C=0.1 scores 0.8534 / 0.6463.

The hybrid produces **16 Funding pseudo-positives**, while K-NN-only recovery produces 45; the lowered cluster threshold previously produced 112. Both best augmented local methods identify 2 of the 4 Funding-positive validation trials, with no Funding false positives (precision 1.0, recall 0.5, F1 0.6667). Direct K-NN performs differently: its best configuration uses 3 distance-weighted neighbors and scores sample F1 0.8632, but identifies no Funding-positive validation trials.

This shows that local recovery can preserve rare-label information without promoting an entire cluster in this experiment. It does not prove general superiority: 78 validation trials, four Funding positives, and repeated selection across 99 total evaluated configurations (70 initial, 4 Funding thresholds, 25 neighbor controls/candidates) introduce selection uncertainty. K-NN-only recovery has higher macro F1 than the hybrid; the hybrid is chosen among the new augmented candidates because sample-wise F1 is the stated primary metric.

The original submission/model artifacts are frozen for comparison. The new hybrid candidate, recovered labels, all validation predictions, and settings are saved under `part1-results/knn-experiment/`. Clustering remains part of the hybrid, satisfying the unsupervised-method requirement; direct K-NN by itself would not. No test score or Kaggle submission is claimed.

### Save Part 1 predictions

Save the selected semi-supervised model's test predictions as `ID,Predicted`. Reason names are space-separated. These are local files; this notebook does not upload them to Kaggle.

In [13]:
if RUN_FULL:
    test_pred = selected_test_pred
    predicted_reasons = []

    for row in test_pred:
        names = [label for label, value in zip(labels, row) if value == 1]
        predicted_reasons.append(" ".join(names))

    part1_submission = reason_test[["ID"]].copy()
    part1_submission["Predicted"] = predicted_reasons
    part1_submission.to_csv(WORK / "part1_submission.csv", index=False)
    part1_submission.head()

    assert part1_submission.ID.equals(reason_test.ID)
    assert part1_submission.Predicted.str.len().gt(0).all()
    assert len(part1_submission) == 312
else:
    print("Part 1 original cell 22 not retrained in verification mode.")


Part 1 original cell 22 not retrained in verification mode.


## Verify the final hybrid and distinguish the earlier submission

The final submission is the knn-experiment hybrid, not the root K-Means submission. Reported public F1 0.87927 is recorded in PART1_NOTES.md and was not re-queried.


In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
reasons = ['Safety','Funding','Efficacy','Recruitment','Other']
truth = pd.read_csv(ROOT / 'part1-data/part1_val.csv')
artifacts = WORK / 'part1-results' if RUN_FULL else ROOT / 'part1-results'
validation = pd.read_csv(artifacts / 'knn-experiment/validation_kmeans_knn_hybrid_k3_uniform.csv')
assert validation.ID.equals(truth.ID)
metrics = pd.Series({'sample_f1':f1_score(truth[reasons],validation[reasons],average='samples'),
                     'macro_f1':f1_score(truth[reasons],validation[reasons],average='macro')})
assert np.isclose(metrics.sample_f1,.8867521367521366)
assert np.isclose(metrics.macro_f1,.7792343604108309)
display(metrics.to_frame('validation'))
submission = pd.read_csv(artifacts / 'knn-experiment/best_augmented_candidate_submission.csv')
test = pd.read_csv(ROOT / 'part1-data/part1_test.csv')
assert len(submission)==312 and submission.ID.equals(test.ID)
assert submission.Predicted.notna().all() and submission.Predicted.str.strip().ne('').all()
assert all(set(s.split()) <= set(reasons) for s in submission.Predicted)
display(pd.read_csv(artifacts / 'knn-experiment/results.csv').groupby('method',sort=False).head(1))


,validation
sample_f1,0.886752
macro_f1,0.779234


,method,neighbors,neighbor_weighting,funding_precision,funding_recall,funding_tp,funding_fp,funding_fn,sample_f1,macro_f1,Safety,Funding,Efficacy,Recruitment,Other,pseudo_Safety,pseudo_Funding,pseudo_Efficacy,pseudo_Recruitment,pseudo_Other
0,kmeans_knn_hybrid,3,uniform,1.000,0.50,2,0,2,0.886752,0.779234,0.888889,0.666667,0.800000,0.952381,0.588235,95,16,147,766,112
1,knn_recovery,3,uniform,1.000,0.50,2,0,2,0.880342,0.787296,0.900000,0.666667,0.785714,0.917431,0.666667,129,45,163,838,222
4,direct_knn,3,distance,0.000,0.00,0,0,4,0.863248,0.613284,0.800000,0.000000,0.769231,0.970874,0.526316,0,0,0,0,0
8,supervised_lr_C0.1,0,none,0.125,0.25,1,7,3,0.853419,0.646342,0.750000,0.166667,0.742857,0.979592,0.592593,0,0,0,0,0


## Findings

Initial cluster augmentation underperformed the supervised control. The equal K-Means/local-neighbor blend reaches validation sample F1 0.88675 and macro F1 0.77923; its 99-setting selection and rare labels limit confidence. PCA is diagnostic only. These results belong to Part 1; no reason embedding or reason prediction enters the Part 2 classifier.
